# Phase 5 — 3-way 3D bake-off (cups vs meridian vs thickrib)

Picks the V1 blade by **3D ground truth**, not any single campaign's 2D-slice ranking. Rebuilds
a combined field from the three campaigns' Drive paretos (top-5 cups + top-20 meridian + top-20
thickrib = 45 designs), 3D-verifies **all** of them with the same pipeline, and ranks by 3D
`J_fan` — tagged by source so you see which iteration actually wins.

Expensive (~50 min/design; ~5 h on 8 cores). Writes `verification.json` incrementally to Drive,
so a disconnect resumes. Tiny scratch (~11 MB/design).

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "blade-redesign-aero-first"
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery", "plotly"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Imports

In [ ]:
import json
import numpy as np
from fanopt.geometry.blade import BladeParams
from fanopt.cfd.blade_verify import verify_blades
from fanopt.cfd.phase5 import VerifyConfig
from fanopt.utils.ledger import design_hash

## 3. Build the combined field from the three Drive campaigns

Top-K from each pareto (edit the counts if you want). Cups use the pre-enrichment schema —
`BladeParams.from_dict` loads them via back-compat. Each design is tagged with its `source`.

In [ ]:
SOURCES = {
    "cups":     (DRIVE_ROOT / "phase4_aero" / "pareto.json", 5),
    "meridian": (DRIVE_ROOT / "phase4_aero_meridian" / "pareto.json", 20),
    "thickrib": (DRIVE_ROOT / "phase4_aero_thickrib" / "pareto.json", 20),
}
combined = []
for name, (path, k) in SOURCES.items():
    pareto = json.loads(Path(path).read_text())
    top = sorted(pareto, key=lambda d: d["j_fan"], reverse=True)[:k]
    for d in top:
        d = dict(d); d["source"] = name
        combined.append(d)
    print(f"{name:10s}: {len(pareto):3d} pareto -> top-{len(top)}")
src_of = {design_hash(d["params"]): d["source"] for d in combined}
print(f"combined field: {len(combined)} designs to 3D-verify")

## 4. Run the 3D verify (all 45) — the ~5 h step

`verify_blades` builds each blade from its absolute params → STEP → 3D mesh → unsteady SU2 →
cycle-mean 3D `J_fan`. Writes `verification.json` to Drive after every design (resumable).

In [ ]:
N_WORKERS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
OUT = DRIVE_ROOT / "phase5_bakeoff"; OUT.mkdir(parents=True, exist_ok=True)
_vf = OUT / "verification.json"

# IDEMPOTENT: if the bake-off already finished on Drive, skip the ~5 h run (safe to "Run all").
_complete = _vf.exists() and sum(
    d.get("j_fan_3d") is not None for d in json.loads(_vf.read_text()).get("designs", [])) >= len(combined)
if _complete:
    print(f"bake-off already complete on Drive ({len(combined)} designs) -> SKIPPING the ~5 h run.")
else:
    def _summary(results):
        return {"designs": [{"name": r.name, "j_fan_3d": r.j_fan_3d, "j_fan_slice": r.j_fan_slice,
                             "n_nodes": r.meta.get("n_nodes")} for r in results]}
    done = []
    def _ckpt(r):
        done.append(r); _vf.write_text(json.dumps(_summary(done), indent=2))
    print(f"verifying {len(combined)} designs on {N_WORKERS} workers (~5 h)...")
    results, _ = verify_blades(combined, OUT, top_k=None, cfg=VerifyConfig(),
                               su2_bin=SU2_BIN, n_workers=N_WORKERS, progress=True, on_result=_ckpt)
    _vf.write_text(json.dumps(_summary(results), indent=2))
    print("done:", len(results), "verified")

## 5. Ranked 3-way results — who wins in 3D?

Sorted by 3D `J_fan`, tagged by source. The 3D values tend to cluster (~7% spread — an aero
plateau), so if the top few are within noise, re-verify just those at high fidelity
(`VerifyConfig(n_cycles=5, inner_iter=100)`) to separate them before promoting to TO.

In [ ]:
# reads verification.json off Drive, so it works whether cell 4 ran or was skipped
vj = json.loads((DRIVE_ROOT / "phase5_bakeoff" / "verification.json").read_text())
rows = []
for d in vj["designs"]:
    if d["j_fan_3d"] is None: continue
    h = d["name"].split("_", 1)[1]
    rows.append((d["j_fan_3d"], d["j_fan_slice"], src_of.get(h, "?"), d["name"]))
rows.sort(reverse=True)
print(f"{'rank':>4} {'3D J_fan':>11} {'slice J_fan':>11} {'source':>10}")
for i, (j3, js, src, _n) in enumerate(rows[:15], 1):
    print(f"{i:>4} {j3:11.3e} {js:11.3e} {src:>10}")
j3s = np.array([r[0] for r in rows])
print(f"\n3D J_fan spread: {j3s.min():.2e} .. {j3s.max():.2e}  "
      f"({(j3s.max()-j3s.min())/j3s.mean()*100:.0f}% of mean)")
from collections import Counter
print("top-10 by source:", dict(Counter(r[2] for r in rows[:10])))

## 6. High-fidelity tiebreak — confirm the top-5 (5 cyc / 100 inner)

Demo fidelity (3 cyc / 30 inner) can leave the unsteady force under-converged — a 2× outlier
especially needs confirming. This re-verifies just the **top-5 by bake-off 3D `J_fan`** at high
fidelity, to separate a real champion from a fidelity artifact. **Self-contained:** run cells
1–3 (setup + imports + build the field) then this — you do **not** need to re-run the ~5 h
bake-off (cell 4); it reads those results off Drive.

In [ ]:
import os
N_WORKERS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
by_hash = {design_hash(d["params"]): d for d in combined}
bake = json.loads((DRIVE_ROOT / "phase5_bakeoff" / "verification.json").read_text())
ranked = sorted([z for z in bake["designs"] if z["j_fan_3d"] is not None],
                key=lambda z: z["j_fan_3d"], reverse=True)[:5]
top5 = [by_hash[z["name"].split("_", 1)[1]] for z in ranked]
print("top-5 by bake-off 3D J_fan -> re-verify at HIGH fidelity:")
for z, d in zip(ranked, top5):
    print(f"  {d['source']:9s}  bakeoff 3D {z['j_fan_3d']:.3e}")

TB = DRIVE_ROOT / "phase5_tiebreak"; TB.mkdir(parents=True, exist_ok=True)
_tvf = TB / "verification.json"
_tdone = _tvf.exists() and sum(
    z.get("j_fan_3d") is not None for z in json.loads(_tvf.read_text()).get("designs", [])) >= len(top5)
if _tdone:
    print("tiebreak already complete on Drive -> loading (skip the ~2-4 h re-verify).")
else:
    tb = []
    def _tck(r):
        tb.append(r); _tvf.write_text(json.dumps(
            {"designs": [{"name": x.name, "j_fan_3d": x.j_fan_3d, "j_fan_slice": x.j_fan_slice}
                         for x in tb]}, indent=2))
    verify_blades(top5, TB, top_k=None, cfg=VerifyConfig(n_cycles=5, inner_iter=100),
                  su2_bin=SU2_BIN, n_workers=min(5, N_WORKERS), progress=True, on_result=_tck)

src_h = {design_hash(d["params"]): d["source"] for d in top5}
tv = json.loads(_tvf.read_text())
print("\n=== HIGH-FIDELITY 3D J_fan (5cyc/100inner) — the tiebreak result ===")
for z in sorted([d for d in tv["designs"] if d["j_fan_3d"] is not None],
                key=lambda z: z["j_fan_3d"], reverse=True):
    print(f"  {z['j_fan_3d']:11.3e}  {src_h.get(z['name'].split('_',1)[1],'?'):9s}  slice {z['j_fan_slice']:.3e}")